In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import(
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
from sklearn.pipeline import Pipeline

import pandas as pd

In [ ]:
DATA_DIR = Path("../data/processed")

input_file = DATA_DIR / "modeling_data.joblib"

modeling_data = joblib.load(input_file)

print("Données chargées avec succès.")
print("Chemin :", input_file.resolve())

Données chargées avec succès.
Chemin : C:\Users\ajbba\Desktop\YOUCODE-IA\Mental_Health_Risk_Prediction\data\processed\modeling_data.joblib
{'X_train': array([[-0.0637065 , -0.47163633, -1.40418192, ...,  0.        ,
         1.        ,  0.        ],
       [ 0.87574418, -1.2323401 , -1.6760358 , ...,  0.        ,
         1.        ,  0.        ],
       [-1.00315718, -0.47163633,  0.40817727, ...,  0.        ,
         1.        ,  0.        ],
       ...,
       [-1.94260785,  1.81047496,  0.04570543, ...,  1.        ,
         1.        ,  0.        ],
       [ 0.40601884,  1.04977119, -0.7698562 , ...,  0.        ,
         1.        ,  0.        ],
       [-0.53343184, -1.2323401 , -0.95109212, ...,  0.        ,
         0.        ,  1.        ]], shape=(1600, 19)), 'X_test': array([[ 1.34546952,  1.81047496, -1.40418192, ...,  0.        ,
         0.        ,  1.        ],
       [-0.0637065 ,  0.28906743,  0.22694135, ...,  0.        ,
         0.        ,  1.        ],
       

In [4]:
X_train = modeling_data["X_train"]
X_test = modeling_data["X_test"]

y_train = modeling_data["y_train"]
y_test = modeling_data["y_test"]

print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test :", X_test.shape)

print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test :", y_test.shape)

Dimensions de X_train : (1600, 19)
Dimensions de X_test : (400, 19)
Dimensions de y_train : (1600,)
Dimensions de y_test : (400,)


Entraîner au moins 3 modèles de classification.

In [6]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    "KNN": KNeighborsClassifier(
        n_neighbors=5
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=42
    )
}

print("Les trois models ont été créés avec succes")

Les trois models ont été créés avec succes


In [11]:
trained_models={}
for model_name,model in models.items():
    print("entrainement de model:",model_name)
    model.fit(X_train,y_train)
    trained_models[model_name]=model
    print("entrainement terminee!")

entrainement de model: Logistic Regression
entrainement terminee!
entrainement de model: KNN
entrainement terminee!
entrainement de model: Random Forest
entrainement terminee!


In [13]:
print("Nombre de modèles entraînés :", len(trained_models))

for model_name in trained_models:
    print("*", model_name)

Nombre de modèles entraînés : 3
* Logistic Regression
* KNN
* Random Forest


Utiliser les mêmes données d’entraînement pour les comparer.

In [16]:
predictions={}
for model_name,model in trained_models.items():
    y_pred=model.predict(X_test)
    predictions[model_name]=y_pred
    print("predection terminee pour le model",model_name)

predection terminee pour le model Logistic Regression
predection terminee pour le model KNN
predection terminee pour le model Random Forest


Évaluer les modèles avec des métriques adaptées :accuracy, précision, rappel, F1-score, matrice de confusion et, si pertinent, ROC-AUC.


In [17]:
results = []

for model_name, model in trained_models.items():

    y_pred = predictions[model_name]

    y_proba = model.predict_proba(X_test)

    accuracy = accuracy_score(y_test, y_pred)

    precision = precision_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        y_proba,
        multi_class="ovr",
        average="macro",
        labels=model.classes_
    )

    results.append({
        "Model": model_name,
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1-score": f1,
        "ROC-AUC": roc_auc
    })

print("Évaluation des modèles terminée.")

Évaluation des modèles terminée.


In [18]:
results_df = pd.DataFrame(results)

results_df = results_df.set_index("Model")

display(results_df.round(4))

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Model,,,,,
Logistic Regression,0.7575,0.7570,0.7568,0.7569,0.9140
KNN,0.5850,0.5691,0.5840,0.5734,0.7695
Random Forest,0.7025,0.6995,0.7022,0.7007,0.8624


In [19]:
for model_name, model in trained_models.items():

    y_pred = predictions[model_name]

    cm = confusion_matrix(
        y_test,
        y_pred,
        labels=model.classes_
    )

    cm_df = pd.DataFrame(
        cm,
        index=model.classes_,
        columns=model.classes_
    )

    print(f"\nConfusion Matrix : {model_name}")

    display(cm_df)


Confusion Matrix : Logistic Regression


,High,Low,Medium
High,112,0,24
Low,1,107,24
Medium,25,23,84



Confusion Matrix : KNN


,High,Low,Medium
High,93,9,34
Low,9,97,26
Medium,41,47,44



Confusion Matrix : Random Forest


,High,Low,Medium
High,100,3,33
Low,3,108,21
Medium,33,26,73


In [20]:
for model_name, model in trained_models.items():

    y_pred = predictions[model_name]

    print(f"\nClassification Report : {model_name}")

    print(
        classification_report(
            y_test,
            y_pred,
            labels=model.classes_,
            zero_division=0
        )
    )


Classification Report : Logistic Regression
              precision    recall  f1-score   support

        High       0.81      0.82      0.82       136
         Low       0.82      0.81      0.82       132
      Medium       0.64      0.64      0.64       132

    accuracy                           0.76       400
   macro avg       0.76      0.76      0.76       400
weighted avg       0.76      0.76      0.76       400


Classification Report : KNN
              precision    recall  f1-score   support

        High       0.65      0.68      0.67       136
         Low       0.63      0.73      0.68       132
      Medium       0.42      0.33      0.37       132

    accuracy                           0.58       400
   macro avg       0.57      0.58      0.57       400
weighted avg       0.57      0.58      0.57       400


Classification Report : Random Forest
              precision    recall  f1-score   support

        High       0.74      0.74      0.74       136
         Low    

In [21]:
ranking = results_df.sort_values(
    by="F1-score",
    ascending=False
)

display(ranking.round(4))

,Accuracy,Precision,Recall,F1-score,ROC-AUC
Model,,,,,
Logistic Regression,0.7575,0.7570,0.7568,0.7569,0.9140
Random Forest,0.7025,0.6995,0.7022,0.7007,0.8624
KNN,0.5850,0.5691,0.5840,0.5734,0.7695
